# Model: will the pitch be called a strike?

In this notebook I estimate **P(strike)**, the chance that a pitch the batter takes is called a strike. I learn from the pitches in `pitches.csv`, where I know the call, and then predict every pitch in `predict.csv`. Predictions are saved in `strike_predictions.csv`.

## Setup

- `HALF_PLATE`: half the width of home plate (17 inches), in feet
- `BALL_RADIUS`: Takes into account the radius of the ball since the pitch only has to touch the zone rather than be fully contained in it

`FEATURES` is the feature list as shown here:

| Feature | What it means |
|---|---|
| `inches_outside` | How far the edge of the ball is from the zone, in inches. 0 means the ball just touches the zone, and negative means it's over the zone |
| `three_balls` | 1 if the count has three balls, otherwise 0 |
| `two_strikes` | 1 if the count has two strikes, otherwise 0 |

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

HALF_PLATE = 17 / 12 / 2
BALL_RADIUS = 1.45 / 12
FEATURES = ["inches_outside", "three_balls", "two_strikes"]

## Load the pitches

I put both files into one table. `known` marks the pitches from `pitches.csv`, where I know the call.

I fix two problems in `pitches.csv` first:

- 12 rows are exact copies of another row, with the same `pitch_id` and every other value. I keep one copy so those pitches don't count twice.
- 5 rows have a blank `is_strike`. Those rows were dropped.

In [2]:
labeled = pd.read_csv("../pitches.csv").drop_duplicates().dropna(subset=["is_strike"])
pitches = pd.concat([labeled, pd.read_csv("../predict.csv")], ignore_index=True)
known = pitches.is_strike.notna()

print("pitches with a known call:", known.sum())
print("pitches to predict:", (~known).sum())

pitches with a known call: 3732
pitches to predict: 901


## Fix the strike zone in one game

Had some issues before noticing that one of the games has strike zone measurements in inches rather than feet. I fixed that by dividing by 12 which solved the issue.

In [3]:
in_inches = pitches.sz_top > 10
print("games:", pitches.game_pk[in_inches].unique())
print("pitches with a known call:", (in_inches & known).sum())
print("pitches to predict:", (in_inches & ~known).sum())

pitches.loc[in_inches, ["sz_top", "sz_bot"]] /= 12

games: [824918]
pitches with a known call: 107
pitches to predict: 20


## The strike zone in this data is the ABS zone

The zone is based on the batter's height. The top is at 53.5% of his height and the bottom is at 27%, so taller batters get a taller zone. The check below shows that `sz_top` and `sz_bot` follow that rule, so I can measure every pitch against each batter's exact zone.

In [4]:
ratio = pitches.sz_bot / pitches.sz_top

print("zone top, shortest to tallest batter, in inches off the ground:", (12 * pitches.sz_top.min()).round(1), "to", (12 * pitches.sz_top.max()).round(1))
print("sz_bot / sz_top (27 / 53.5 = 0.5047):", ratio.min().round(4), "to", ratio.max().round(4))

zone top, shortest to tallest batter, in inches off the ground: 35.2 to 42.5
sz_bot / sz_top (27 / 53.5 = 0.5047): 0.5046 to 0.5048


## How far is each pitch from the edge of the zone?

`plate_x` and `plate_z` give the location of the center of the ball, in feet. A pitch only has to touch the zone to be a strike, so I measure from the edge of the ball instead: the distance from the center of the ball to the zone, minus the ball's radius.

- **Positive**: how far the ball missed the zone. Pitches off a corner are measured to the corner.
- **0**: the ball just touches the zone.
- **Negative**: the ball is over the zone, and the number is how far it would have to move to miss it.

`past_side` is how far the center of the ball is past the left or right edge, and `past_top_or_bottom` is how far it is above or below the zone. Both are negative when the center of the ball is inside those edges.

In [5]:
past_side = 12 * (pitches.plate_x.abs() - HALF_PLATE)
past_top_or_bottom = 12 * np.maximum(pitches.plate_z - pitches.sz_top, pitches.sz_bot - pitches.plate_z)

center_outside = np.hypot(past_side.clip(lower=0), past_top_or_bottom.clip(lower=0))
center_inside = np.maximum(past_side, past_top_or_bottom)
pitches["inches_outside"] = center_outside.where(center_outside > 0, center_inside) - 12 * BALL_RADIUS

## The count

Umpires usually call a smaller zone with two strikes and a bigger zone with three balls. I give the model a yes/no column for each. A 3-2 count gets a 1 in both.

In [6]:
pitches["three_balls"] = (pitches.balls == 3).astype(int)
pitches["two_strikes"] = (pitches.strikes == 2).astype(int)

## How closely do the calls follow the zone?

`train` holds the pitches with a known call. The table groups them by how far they are from the zone, 1 inch at a time.

In [7]:
train = pitches[known].astype({"is_strike": int})
train["in_zone"] = (train.inches_outside <= 0).astype(int)

print("calls that match the rulebook zone:", (train.in_zone == train.is_strike).mean().round(3))
print("furthest called strike outside the zone, in inches:", train.inches_outside[train.is_strike == 1].max().round(1))
print("deepest ball inside the zone, in inches:", -train.inches_outside[train.is_strike == 0].min().round(1))

distance = pd.cut(train.inches_outside, [-np.inf, -3, -2, -1, 0, 1, 2, 3, np.inf])
train.groupby(distance, observed=True).is_strike.agg(pitches="size", strike_rate="mean").round(3)

calls that match the rulebook zone: 0.958
furthest called strike outside the zone, in inches: 3.4
deepest ball inside the zone, in inches: 2.7


,pitches,strike_rate
inches_outside,,
"(-inf, -3.0]",588,1.000
"(-3.0, -2.0]",171,0.982
"(-2.0, -1.0]",216,0.949
"(-1.0, 0.0]",198,0.707
"(0.0, 1.0]",194,0.278
"(1.0, 2.0]",200,0.110
"(2.0, 3.0]",210,0.029
"(3.0, inf]",1955,0.001


Very closely. About 96% of calls match the rulebook, and every missed call is within a few inches of the edge. More than 3 inches inside the zone is always a strike, and more than 3 inches outside is almost never one. In between, the strike rate drops smoothly from about 98% to about 3%.

That S-shaped drop is what a logistic regression models, so that's the model I use. It only has to learn where the 50% point is, how fast the chance drops around it, and how much the count moves it.

## How good is the model?

To grade the model fairly, every pitch needs a prediction from a model that didn't see it during training. I use *cross-validation*: split the known pitches into 5 random groups, train on 4, predict the 5th, and repeat for each group. `predict.csv` has about 20% of the pitches from each of the same 30 games, so a random split matches the real task.

I compare my model with the obvious answer, the rulebook zone. That model gives one chance to every pitch in the zone and another to every pitch outside it.

Predictions are graded by two metrics:

1. **Log loss**: grades the chances themselves. Confident misses are punished the most, so calling a ball 99% strike costs much more than calling it 60%. Lower is better and 0 is perfect.
2. **AUC**: take one strike and one ball. How often does the model give the strike the higher chance? 0.5 is a coin flip and 1.0 is perfect.

In [8]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)


def cv_chances(columns):
    return cross_val_predict(LogisticRegression(), train[columns], train.is_strike, cv=folds, method="predict_proba")[:, 1]


train["rulebook_chance"] = cv_chances(["in_zone"])
train["cv_chance"] = cv_chances(FEATURES)

grades = {}
for name, chance in [("rulebook zone", train.rulebook_chance), ("my model", train.cv_chance)]:
    grades[name] = {"log loss": log_loss(train.is_strike, chance), "AUC": roc_auc_score(train.is_strike, chance)}
pd.DataFrame(grades).T.round(3)

,log loss,AUC
rulebook zone,0.172,0.944
my model,0.104,0.992


My model's log loss is 0.104, compared to 0.172 for the rulebook zone. It also ranks strikes above balls more often (AUC of 0.992 vs 0.944). How far a pitch is from the edge matters much more than just whether it's in or out.

I also tried adding pitch type, batter side and pitcher hand, catcher, and umpire, and I tried gradient boosting like in my baserunning model. None of them made the model meaningfully better. Taking out either count flag made it slightly worse, so I kept both.

## Do the predicted chances match what happened?

AUC only checks the order of the chances, so I also check the chances themselves. I group pitches by predicted chance and compare expected strikes (number of pitches x average predicted chance) with actual strikes. If the chances are right, the two should be close.

Most pitches are easy calls near 0% or 100%, so I use fixed ranges instead of equal-sized groups. That gives the close calls their own rows.

In [9]:
groups = pd.cut(train.cv_chance, [0, 0.02, 0.1, 0.3, 0.5, 0.7, 0.9, 0.98, 1], include_lowest=True)
calibration = train.groupby(groups, observed=True).agg(pitches=("is_strike", "size"),
                                                        predicted_strike=("cv_chance", "mean"),
                                                        actual_strike=("is_strike", "mean"))
calibration["expected strikes"] = calibration.pitches * calibration.predicted_strike
calibration["actual strikes"] = calibration.pitches * calibration.actual_strike
calibration.round(3)

,pitches,predicted_strike,actual_strike,expected strikes,actual strikes
cv_chance,,,,,
"(-0.001, 0.02]",2070,0.001,0.001,2.364,2.0
"(0.02, 0.1]",226,0.050,0.066,11.289,15.0
"(0.1, 0.3]",163,0.191,0.160,31.156,26.0
"(0.3, 0.5]",80,0.398,0.412,31.815,33.0
"(0.5, 0.7]",106,0.599,0.594,63.513,63.0
"(0.7, 0.9]",164,0.811,0.811,133.014,133.0
"(0.9, 0.98]",199,0.949,0.955,188.933,190.0
"(0.98, 1.0]",724,0.997,0.997,721.898,722.0


## Train the final model and predict

I train on every known pitch and predict the pitches in `predict.csv`. `predict_proba` gives two chances for each pitch, ball and strike. I keep `[:, 1]`, the chance of a strike.

In [10]:
model = LogisticRegression()
model.fit(train[FEATURES], train.is_strike)
pitches.loc[~known, "prob_strike"] = model.predict_proba(pitches.loc[~known, FEATURES])[:, 1]

## What did the model learn?

The table shows the model's chance of a strike near the edge of the zone. Each column is inches from the edge, where 0 means the ball just touches the zone. "Three balls" is 3-0 or 3-1, and "two strikes" is 0-2, 1-2 or 2-2.

In [11]:
examples = {}
for count, three_balls, two_strikes in [("other counts", 0, 0), ("two strikes", 0, 1), ("three balls", 1, 0)]:
    X = pd.DataFrame({"inches_outside": [-2, -1, 0, 1, 2], "three_balls": three_balls, "two_strikes": two_strikes})
    examples[count] = pd.Series(model.predict_proba(X)[:, 1], index=X.inches_outside)
pd.DataFrame(examples).T.round(2)

inches_outside,-2,-1,0,1,2
other counts,0.97,0.87,0.57,0.20,0.05
two strikes,0.96,0.80,0.43,0.13,0.03
three balls,0.95,0.77,0.38,0.11,0.02


- A pitch right on the edge is a strike a little over half the time. One inch in, it's usually a strike, and one inch out, it usually isn't. Umpires are calling a very accurate zone, and nearly all of the uncertain calls are within 2 inches of it.
- With two strikes, the zone gets a little smaller.
- With three balls, the zone also gets smaller which was a little surprising. This could be a result of a smaller dataset, but there isn't a great way to know without more data to test on.

## Save

I saved `pitch_id` and `prob_strike` for every pitch in `predict.csv`, then check for one row per pitch and predict every pitch between 0 and 1.

The average predicted chance is a little lower than the strike rate in `pitches.csv`. That's mostly because fewer of the pitches in `predict.csv` are in the zone.

In [12]:
predictions = pitches.loc[~known, ["pitch_id", "prob_strike"]]
predictions.to_csv("../strike_predictions.csv", index=False)

print("rows:", len(predictions), "of", len(pd.read_csv("../predict.csv")))
print("blank chances:", predictions.prob_strike.isna().sum())
print("every chance between 0 and 1:", predictions.prob_strike.between(0, 1).all())
print()
print("average chance of a strike:", predictions.prob_strike.mean().round(3))
print("strike rate in pitches.csv:", train.is_strike.mean().round(3))
print("share in the zone, pitches to predict:", (pitches.inches_outside[~known] <= 0).mean().round(3))
print("share in the zone, pitches.csv:", train.in_zone.mean().round(3))

rows: 901 of 901
blank chances: 0
every chance between 0 and 1: True

average chance of a strike: 0.291
strike rate in pitches.csv: 0.317
share in the zone, pitches to predict: 0.283
share in the zone, pitches.csv: 0.314
